In [1]:
import sys

sys.path.append("..")

In [2]:
from tqdm import tqdm
import numpy as np
import os
from matplotlib import pyplot as plt
from os.path import join as pjoin
from omegaconf import OmegaConf

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

from utils.misc import load_config
from datasets.data_preparation import prepare_data
from utils.misc import plot_loss_curve, current_time

In [3]:
exp_path = '../results/policy/dl.vit/2023-08-08 12Hr 53Min 41Sec IST+0530'

config = load_config('.', exp_path, 'hyperparameters.yaml')

In [4]:
trainer_config = config['trainer']
data_config = config['data']

checkpoint_name = trainer_config['checkpoint_name']
device = trainer_config['device']
batch_size = trainer_config['batch_size']

patch_size = data_config['patch']['patch_size']
lithology_classes = data_config['lithology_classes']

config['root'] = '..'

In [5]:
# # Load the Libraries
# from sklearn.preprocessing import StandardScaler
# def scale_data(X, data_config):
#     if 'lat' in data_config['scaled_columns']:
#         # lat_min, lat_max = X.lat.min(), X.lat.max()
#         # X.lat = (X.lat - lat_min) / (lat_max - lat_min)
#         scaler_lat = StandardScaler()
#         X.lat = scaler_lat.fit_transform(X.lat.values.reshape(-1, 1))

#     if 'lng' in data_config['scaled_columns']:
#         # lng_min, lng_max = X.lng.min(), X.lng.max()
#         # X.lng = (X.lng - lng_min) / (lng_max - lng_min)
#         scaler_lng = StandardScaler()
#         X.lng = scaler_lng.fit_transform(X.lng.values.reshape(-1, 1))

#     if 'DEPT' in data_config['scaled_columns']:
#         # depth_min, depth_max = X.DEPT.min(), X.DEPT.max()
#         # X.DEPT = (X.DEPT - depth_min) / (depth_max - depth_min)
#         scaler_depth = StandardScaler()
#         X.DEPT = scaler_depth.fit_transform(X.DEPT.values.reshape(-1, 1))

#     if 'ILD' in data_config['scaled_columns']:
#         scaler_ild = StandardScaler()
#         X.ILD = scaler_ild.fit_transform(X.ILD.values.reshape(-1, 1))

#     if 'GR' in data_config['scaled_columns']:
#         scaler_gr = StandardScaler()
#         X.GR = scaler_gr.fit_transform(X.GR.values.reshape(-1, 1))

#     if 'NPHI' in data_config['scaled_columns']:
#         scaler_nphi = StandardScaler()
#         X.NPHI = scaler_nphi.fit_transform(X.NPHI.values.reshape(-1, 1))

#     if 'DPHI' in data_config['scaled_columns']:
#         scaler_dphi = StandardScaler()
#         X.DPHI = scaler_dphi.fit_transform(X.DPHI.values.reshape(-1, 1))

#     # uwi = X.UWI
#     # X = X.drop(['UWI'], axis=1)
#     # cols = X.columns

#     # scaler = StandardScaler()
#     # X = scaler.fit_transform(X)

#     # X = pd.DataFrame(X)
#     # X.columns = cols
#     # X = pd.concat([X.reset_index(drop=True), uwi.reset_index(drop=True)], axis = 1)
    
#     return X

In [ ]:
x_train, x_val, _, _, _ = prepare_data(config)

# x_train = x_train.permute(0, 2, 1)
# x_val = x_val.permute(0, 2, 1)

In [ ]:
traindataset = TensorDataset(x_train)
trainloader = DataLoader(traindataset, batch_size=batch_size, shuffle=True)

valdataset = TensorDataset(x_val)
valloader = DataLoader(valdataset, batch_size=batch_size, shuffle=False)

In [ ]:
class check(nn.Module):
    def __init__(self):
        super(check, self).__init__()
        self.linear = nn.Linear(1, 1)
    def forward(self, x):
        print('shit')
        return x

In [ ]:
from model.vit import ViTEncoder
class Autoencoder(nn.Module):
    def __init__(self, encoder, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = encoder
        self.decoder = nn.Sequential(
            nn.Linear(encoder.to_latent[2].in_features, encoder.to_latent[2].out_features),
            nn.LayerNorm(encoder.to_latent[2].out_features),
            nn.ReLU(),
            nn.Linear(encoder.to_latent[2].out_features, encoder.to_latent[2].out_features//2),
            nn.LayerNorm(encoder.to_latent[2].out_features//2),
            nn.ReLU(),
            nn.Linear(encoder.to_latent[2].out_features//2, encoder.to_latent[2].out_features//2//2),
            nn.LayerNorm(encoder.to_latent[2].out_features//2//2),
            nn.ReLU(),
            nn.Linear(encoder.to_latent[2].out_features//2//2, input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
def train_engine(
        epoch, model, 
        train_loader, 
        criterion, 
        optimizer, 
        num_epochs, 
        device
    ):

    train_loss = 0.0

    model.train()
    for i, batch_inputs in enumerate(tqdm(train_loader, total=len(train_loader), desc=f"Train - Epoch {epoch}/{num_epochs}")):
        optimizer.zero_grad()

        batch_inputs = batch_inputs[0].to(device)

        # Forward pass
        outputs = model(batch_inputs)
        # if i == 1:
        #     plt.plot(batch_inputs[16, 0, :].cpu().numpy(), label="input")
        #     plt.plot(outputs[16, 0, :].detach().cpu().numpy(), label="output")
        #     plt.legend()
        #     plt.title("Train")
        #     plt.show()
        # loss_depth_lat_long = criterion(outputs[:, :, [0, -2, -1]], batch_inputs[:, :, [0, -2, -1]])
        # loss_std = criterion(outputs[:, :, [0, -2, -1]].std(), batch_inputs[:, :, [0, -2, -1]].std())
        # loss_min = criterion(outputs[:, :, [0, -2, -1]].min(), batch_inputs[:, :, [0, -2, -1]].min())
        # loss_max = criterion(outputs[:, :, [0, -2, -1]].max(), batch_inputs[:, :, [0, -2, -1]].max())
        # loss_diff = criterion(outputs[:, :149, [0, -2, -1]] - outputs[:, 1:, [0, -2, -1]],
        #                         batch_inputs[:, :149, [0, -2, -1]] - batch_inputs[:, 1:, [0, -2, -1]])
        # loss_rest = criterion(outputs[:, :, 1:-2], batch_inputs[:, :, 1:-2])
        # loss = (loss_depth_lat_long*3) + (loss_std*2) + (loss_min*2) + (loss_max*2) + loss_rest + (loss_diff*2)
        loss = criterion(outputs, batch_inputs)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Calculate average training loss and accuracy for the epoch
    train_loss /= len(train_loader)

    return train_loss

def validation_engine(
        epoch, 
        model, 
        val_loader, 
        criterion, 
        num_epochs, 
        device
    ):
    
    val_loss = 0.0
    
    # Evaluate on the validation set
    model.eval()
    for i, batch_inputs_val in enumerate(tqdm(val_loader, total=len(val_loader), desc=f"Val - Epoch {epoch}/{num_epochs}")):
        
        batch_inputs_val = batch_inputs_val[0].to(device)

        with torch.no_grad():
            val_outputs = model(batch_inputs_val)

        # if i == 1:
        #     plt.plot(batch_inputs_val[16, 0, :].cpu().numpy(), label="input")
        #     plt.plot(val_outputs[16, 0, :].detach().cpu().numpy(), label="output")
        #     plt.legend()
        #     plt.title("Val")
        #     plt.show()
            
        # loss_depth_lat_long = criterion(val_outputs[:, :, [0, -2, -1]], batch_inputs_val[:, :, [0, -2, -1]])
        # loss_std = criterion(val_outputs[:, :, [0, -2, -1]].std(), batch_inputs_val[:, :, [0, -2, -1]].std())
        # loss_min = criterion(val_outputs[:, :, [0, -2, -1]].min(), batch_inputs_val[:, :, [0, -2, -1]].min())
        # loss_max = criterion(val_outputs[:, :, [0, -2, -1]].max(), batch_inputs_val[:, :, [0, -2, -1]].max())
        # loss_diff = criterion(val_outputs[:, :149, [0, -2, -1]] - val_outputs[:, 1:, [0, -2, -1]], 
        #                         batch_inputs_val[:, :149, [0, -2, -1]] - batch_inputs_val[:, 1:, [0, -2, -1]])
        # loss_rest = criterion(val_outputs[:, :, 1:-2], batch_inputs_val[:, :, 1:-2])
        # loss_val = (loss_depth_lat_long*3) + (loss_std*2) + (loss_min*2) + (loss_max*2) + (loss_diff*2) + loss_rest

        loss_val = criterion(val_outputs, batch_inputs_val)
        
        val_loss += loss_val.item()

    val_loss /= len(val_loader)

    return val_loss

def train(
        num_epochs, 
        model, 
        train_loader, 
        val_loader, 
        criterion, 
        optimizer, 
        tolerance,
        device
    ):
    
    train_losses = []
    val_losses = []
    best_loss = np.inf
    best_model_chkpt = None
    best_optim_chkpt = None
    best_epoch = 1

    for epoch in range(1, num_epochs+1):
        train_loss = train_engine(epoch, model, train_loader, criterion, optimizer, num_epochs, device)
        
        val_loss = validation_engine(epoch, model, val_loader, criterion, num_epochs, device)

        train_losses.append(train_loss)

        val_losses.append(val_loss)

        # Print the progress for the current epoch
        print(f"Epoch {epoch}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        (
            best_loss, 
            best_epoch, 
            best_model_chkpt, 
            best_optim_chkpt
        ) = update_best_metrices(
            val_loss, 
            epoch, 
            model, 
            optimizer, 
            best_loss, 
            best_epoch, 
            best_model_chkpt, 
            best_optim_chkpt
        )

        if epoch - best_epoch > tolerance:
            print("Early stopping")
            break

    return train_losses, val_losses, best_epoch, best_loss, best_model_chkpt, best_optim_chkpt


def update_best_metrices(
        val_loss, 
        epoch, 
        model, 
        optim, 
        best_loss, 
        best_epoch, 
        best_model_chkpt, 
        best_optim_chkpt
        ):

    if val_loss < best_loss:
        print(f"Model Performance Improved from epoch no. {best_epoch}")
        best_loss = val_loss
        best_epoch = epoch
        best_model_chkpt = model.state_dict()
        best_optim_chkpt = optim.state_dict()
    return best_loss, best_epoch, best_model_chkpt, best_optim_chkpt

In [ ]:
config['model']['dim'] = 150

In [ ]:
encoder = ViTEncoder(dim=config['model']['dim'], 
                     depth=config['model']['depth'],
                     heads=config['model']['heads'],
                     mlp_dim=config['model']['mlp_dim'],
                     num_features=data_config['num_features'],
                     patch_size=data_config['patch']['patch_size'],
                     channels=config['model']['channels'],
                     dim_head=config['model']['dim_head'],
                     activation=config['model']['activation'],
                     device=device)
model = Autoencoder(encoder, data_config['num_features'])
model = model.to('cuda')

criterion = nn.MSELoss()
# criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
(
    train_losses, 
    val_losses, 
    best_epoch, 
    best_loss, 
    best_model_chkpt, 
    best_optim_chkpt
) = train(num_epochs=1000,
          model=model,
          train_loader=trainloader,
          val_loader=valloader,
          criterion=criterion,
          optimizer=optimizer,
          tolerance=30,
          device=device)

In [ ]:
encoder_weight = encoder.state_dict()
optimizer_weight = optimizer.state_dict()
checkpoint_dict = {
    'encoder_state_dict': encoder_weight,
    'optimizer_state_dict': optimizer_weight,
}
path = pjoin(
    '..', 
    'pretraining', 
    'encoder', 
    config['model']['__name__'],
    current_time(),
)
os.makedirs(path, exist_ok=True)

torch.save(
    checkpoint_dict,
    pjoin(
        path,
        'checkpoint.pt'
    )
)

OmegaConf.save(
    config, 
    pjoin(path, trainer_config['hyperparameters_filename'])
)

In [ ]:
best_epoch

In [ ]:
plot_loss_curve(train_losses, val_losses,save_path='')

In [ ]:
val_inp = valdataset[:64][0]
out_val = model(val_inp.cuda())

In [ ]:
batch_idx = np.random.randint(64)

In [ ]:
log_idx = np.random.randint(val_inp.shape[-1])

# log_idx = 0
plt.plot(val_inp[batch_idx, :, log_idx], label='Original')
plt.plot(out_val[batch_idx, :, log_idx].cpu().detach(), label='Reconstructed')
plt.legend()
plt.title((val_inp[batch_idx, :, log_idx].std().item(), out_val[batch_idx, :, log_idx].cpu().detach().std().item()))
plt.show()

In [ ]:
plt.plot(out_val[batch_idx, :, 0].cpu().detach(), label='0')
plt.plot(out_val[batch_idx, :, 1].cpu().detach(), label='1')
plt.plot(out_val[batch_idx, :, 2].cpu().detach(), label='2')
plt.plot(out_val[batch_idx, :, 3].cpu().detach(), label='3')
plt.plot(out_val[batch_idx, :, 4].cpu().detach(), label='4')
plt.plot(out_val[batch_idx, :, 5].cpu().detach(), label='5')

In [ ]:
plt.plot(val_inp[batch_idx, :, 0], label='0')
plt.plot(val_inp[batch_idx, :, 1], label='1')
plt.plot(val_inp[batch_idx, :, 2], label='2')
plt.plot(val_inp[batch_idx, :, 3], label='3')
plt.plot(val_inp[batch_idx, :, 4], label='4')
plt.plot(val_inp[batch_idx, :, 5], label='5')